# End-to-End MLOps Pipeline — Exploration & Training Notebook

This notebook mirrors the GeeksforGeeks tutorial: load the student dataset, explore it,
preprocess, train a `RandomForestClassifier`, tune hyperparameters with `GridSearchCV`,
track experiments with MLflow, and serialize the final model.

Run this interactively, cell by cell. The same logic is also available as a script in
`../src/train.py` so it can run unattended in CI/CD and Docker.

In [ ]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
from imblearn.over_sampling import SMOTE

import mlflow
import mlflow.sklearn
import joblib
import matplotlib.pyplot as plt

## 1. Load the Dataset

In [ ]:
df = pd.read_csv("../data/data.csv", sep=";")
df.head()

## 2. Basic Exploration

In [ ]:
print(df.shape)
print(df.info())

In [ ]:
df.isnull().sum().sum()  # confirm no missing values

In [ ]:
df["Target"].value_counts()

## 3. Preprocessing

- Drop irrelevant columns
- Separate features/target
- Encode target labels
- One-hot encode `Course`
- Scale numeric features
- Train/test split
- Balance classes with SMOTE

In [ ]:
df = df.drop(columns=["id"], errors="ignore")

X = df.drop(columns=["Target"])
y = df["Target"]

label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y)

X = pd.get_dummies(X, columns=["Course"], drop_first=True)
feature_columns = X.columns.tolist()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

sm = SMOTE(random_state=42)
X_train, y_train = sm.fit_resample(X_train, y_train)

X_train.shape, X_test.shape

## 4. Baseline Model

In [ ]:
model = RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

## 5. Hyperparameter Tuning with MLflow Tracking

In [ ]:
mlflow.set_experiment("Student_Risk_Prediction")

param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [10, 20, None],
    "min_samples_split": [2, 5],
}

grid = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    cv=3,
    scoring="accuracy",
    n_jobs=-1,
)
grid.fit(X_train, y_train)

grid.best_params_, grid.best_score_

In [ ]:
with mlflow.start_run():
    mlflow.log_params(grid.best_params_)
    mlflow.log_metric("best_accuracy", grid.best_score_)
    mlflow.sklearn.log_model(grid.best_estimator_, "model")

## 6. Evaluate the Best Model

In [ ]:
best_model = grid.best_estimator_
y_pred = best_model.predict(X_test)

acc = accuracy_score(y_test, y_pred)
print("Accuracy:", acc)
print(classification_report(y_test, y_pred))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=label_encoder.classes_)
disp.plot(cmap="Blues", xticks_rotation=45)
plt.title(f"Confusion Matrix (accuracy={acc:.3f})")
plt.tight_layout()
plt.show()

## 7. Serialize Artifacts

These are the same artifacts produced by `../src/train.py`, which is what CI/CD and
Docker actually run. Saving them here too lets you inspect/test locally without
leaving the notebook.

In [ ]:
joblib.dump(best_model, "../model.pkl")
joblib.dump(scaler, "../scaler.pkl")
joblib.dump(label_encoder, "../label_encoder.pkl")
joblib.dump(feature_columns, "../feature_columns.pkl")

print("Saved model.pkl, scaler.pkl, label_encoder.pkl, feature_columns.pkl")

## Next Steps

- To serve predictions locally: `uvicorn src.app:app --reload --port 8000` from the project root.
- To containerize: `docker build -t academic-success-predictor .`
- The production pipeline (`src/train.py`) runs this same logic headlessly in GitHub Actions.